# PyGeoFetch — 02: Authentication & Providers

Covers the full credential system and all 22 providers.

### What you'll learn
- Add credentials via Python API, env vars, and CLI
- BUG 1 fix verified: add_credentials dict form works for all auth types
- Manage stored credentials (list, test, remove)
- Sentinel-1C/D constellation support

In [1]:
from pygeofetch import PyGeoFetch
from pygeofetch.core.authenticator import AuthManager
from pygeofetch.providers import list_provider_info, get_free_providers, get_providers_by_capability
import pandas as pd

client = PyGeoFetch(log_level="WARNING")
auth   = AuthManager()
print("Client and AuthManager ready")

Client and AuthManager ready


## 1. Adding Credentials

In [2]:
# Python API — all auth types now use the same interface (BUG 1 fix)

# username / password
# client.add_credentials("usgs", username="USER", password="PASS")
client.add_credentials("copernicus", username="appiahkubis14@gmail.com", password="USGS@sak@2001server")
# client.add_credentials("nasa_earthdata", username="USER", password="PASS")

# API key
# client.add_credentials("planet", api_key="YOUR_KEY")
# client.add_credentials("opentopography", api_key="YOUR_KEY")

# OAuth2 client credentials
# client.add_credentials("sentinel_hub", client_id="ID", client_secret="SECRET")

# Bearer token
# client.add_credentials("maxar_gbdx", token="YOUR_TOKEN")

# print("BUG 1 fix: engine now calls auth.add_credentials(provider, dict)")
# print("All 5 auth types (username/password, api_key, OAuth2, token, access/secret key)")
# print("are accepted through the same add_credentials() interface.")

In [ ]:
# CLI auth commands reference
cli_cmds = [
    "PyGeoFetch auth add usgs --username USER --password PASS",
    "PyGeoFetch auth add copernicus --username email@example.com --password PASS",
    "PyGeoFetch auth add planet --api-key YOUR_KEY",
    "PyGeoFetch auth add sentinel_hub --client-id ID --client-secret SECRET",
    "PyGeoFetch auth login copernicus",
    "PyGeoFetch auth list",
    "PyGeoFetch auth test usgs",
    "PyGeoFetch auth remove planet --yes",
]
for cmd in cli_cmds:
    print(f"  {cmd}")

## 2. Credential Management

In [3]:
entries = auth.list()
if entries:
    print(f"Stored credentials for {len(entries)} provider(s):")
    for e in entries:
        parts = []
        if e.get("has_username"):  parts.append("username")
        if e.get("has_password"):  parts.append("password")
        if e.get("has_api_key"):   parts.append("api_key")
        if e.get("has_token"):     parts.append("token")
        if e.get("has_client_id"): parts.append("client_id+secret")
        print(f"  {e['provider']:<30} {', '.join(parts)}")
else:
    print("No credentials stored yet.")
    print("Set PYGEOFETCH_* env vars or use client.add_credentials()")

Stored credentials for 5 provider(s):
  OpenTopography                 
  copernicus                     
  opentopography                 
  planet                         
  usgs                           


## 3. Sentinel-1C and 1D — Capability 2

In [4]:
from pygeofetch.utils.geo_utils import _normalise_satellite_name

platforms = [
    "SENTINEL-1A", "SENTINEL-1B", "SENTINEL-1C", "SENTINEL-1D",
    "sentinel-1c", "SENTINEL-2A", "SENTINEL-2B",
]
print("Platform normalisation:")
for p in platforms:
    print(f"  {p!r:25} -> {_normalise_satellite_name(p)!r}")

print()
print("All providers now include S1C/S1D in their platform lists:")
s1_list = ["SENTINEL-1A", "SENTINEL-1B", "SENTINEL-1C", "SENTINEL-1D"]
for s in s1_list:
    print(f"  {s}")

Platform normalisation:
  'SENTINEL-1A'             -> 'S1A'
  'SENTINEL-1B'             -> 'S1B'
  'SENTINEL-1C'             -> 'S1C'
  'SENTINEL-1D'             -> 'S1D'
  'sentinel-1c'             -> 'S1C'
  'SENTINEL-2A'             -> 'S2A'
  'SENTINEL-2B'             -> 'S2B'

All providers now include S1C/S1D in their platform lists:
  SENTINEL-1A
  SENTINEL-1B
  SENTINEL-1C
  SENTINEL-1D


## 4. All 22 Providers

In [5]:
providers = list_provider_info()
free_ids  = set(get_free_providers())

rows = []
for p in providers:
    rows.append({
        "ID":          p["id"],
        "Free":        "yes" if p["id"] in free_ids else "—",
        "SAR":         "yes" if p.get("sar") else "—",
        "STAC":        "yes" if p.get("stac") else "—",
        "VHR":         "yes" if p.get("sub_meter") else "—",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

                       ID Free SAR STAC VHR
          airbus_oneatlas    —   —    —   —
alaska_satellite_facility    —   —    —   —
                aws_earth  yes   —  yes   —
               copernicus    —   —    —   —
             digitalglobe  yes   —    —   —
earth_explorer_additional    —   —    —   —
                element84  yes   —  yes   —
               esa_scihub  yes   —    —   —
        geoserver_generic  yes   —    —   —
      google_earth_engine    —   —    —   —
               inpe_cbers  yes   —    —   —
              isro_bhuvan  yes   —    —   —
               jaxa_earth  yes   —    —   —
               maxar_gbdx    —   —    —   —
           nasa_earthdata    —   —    —   —
     nasa_earthdata_cloud    —   —  yes   —
            noaa_big_data  yes   —    —   —
           opentopography    —   —    —   —
                   planet    —   —  yes   —
       planetary_computer  yes   —  yes   —
             sentinel_hub    —   —  yes   —
              terrabotics    —  

## 5. Filter by Capability

In [ ]:
print("SAR providers:")
for pid in get_providers_by_capability(sar=True):      print(f"  {pid}")

print()
print("VHR (<1m) providers:")
for pid in get_providers_by_capability(sub_meter=True): print(f"  {pid}")

print()
print("Open access (no login):")
for pid in get_providers_by_capability(requires_auth=False): print(f"  {pid}")

---
## Summary
- BUG 1: add_credentials() dict form works for all 5 auth types
- Capability 2: S1C/D in all provider platform lists + _normalise_satellite_name()
- Explored all 22 providers and filtered by capability

**Next:** Notebook 03 — Advanced Search